# Phishing Detection - Exploratory Data Analysis

This notebook explores the three datasets used in our multi-modal phishing detection system:
1. **URL dataset** (HuggingFace) - 3,000 URLs with labels
2. **HTML content** (Kaggle) - Raw HTML of each website
3. **Screenshots** (Kaggle) - Rendered webpage images

All three datasets are derived from the same source URLs.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RAW_DIR = Path('../data/raw')
if RAW_DIR.exists():
    print('Datasets available:')
    for d in RAW_DIR.iterdir():
        if d.is_dir():
            print(f'  {d.name}/')
else:
    print(f'WARNING: {RAW_DIR} does not exist. Run scripts/download_data.py first.')

## 1. URL Dataset

In [ ]:
# Load URL dataset
url_csv = RAW_DIR / 'urls' / 'all_urls.csv'
if not url_csv.exists():
    # Try individual split files
    url_parts = list((RAW_DIR / 'urls').glob('*.csv')) if (RAW_DIR / 'urls').exists() else []
    if url_parts:
        url_df = pd.concat([pd.read_csv(f) for f in url_parts], ignore_index=True)
        print(f'Loaded {len(url_parts)} CSV files from urls/')
    else:
        raise FileNotFoundError(
            f'{url_csv} not found. Run: python scripts/download_data.py'
        )
else:
    url_df = pd.read_csv(url_csv)

print(f'Shape: {url_df.shape}')
print(f'Columns: {url_df.columns.tolist()}')
url_df.head(10)

In [ ]:
# Rename columns for consistency
col_map = {}
for col in url_df.columns:
    if col.lower() in ('text', 'url', 'urls'):
        col_map[col] = 'url'
    elif col.lower() in ('labels', 'label'):
        col_map[col] = 'label'
url_df = url_df.rename(columns=col_map)
url_df['label'] = url_df['label'].astype(int)
url_df['label_name'] = url_df['label'].map({0: 'Legitimate', 1: 'Phishing'})
print(url_df.dtypes)
print(f'\nRows: {len(url_df)}')
url_df.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = url_df['label_name'].value_counts()
colors = ['#2ecc71' if c == 'Legitimate' else '#e74c3c' for c in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 20, str(count), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Class Balance')

plt.suptitle('URL Dataset - Label Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# URL length analysis
url_df['url_length'] = url_df['url'].astype(str).str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    subset = url_df[url_df['label'] == label]
    name = 'Legitimate' if label == 0 else 'Phishing'
    axes[0].hist(subset['url_length'], bins=50, alpha=0.6, label=name, color=color, edgecolor='black')

axes[0].set_title('URL Length Distribution')
axes[0].set_xlabel('URL Length (characters)')
axes[0].set_ylabel('Count')
axes[0].legend()

sns.boxplot(data=url_df, x='label_name', y='url_length', ax=axes[1],
            palette={'Legitimate': '#2ecc71', 'Phishing': '#e74c3c'})
axes[1].set_title('URL Length by Class')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print('URL length statistics:')
print(url_df.groupby('label_name')['url_length'].describe().round(1))

In [ ]:
# URL feature extraction
url_str = url_df['url'].astype(str)
url_df['num_dots'] = url_str.str.count(r'\.')
url_df['num_slashes'] = url_str.str.count('/')
url_df['num_hyphens'] = url_str.str.count('-')
url_df['num_digits'] = url_str.str.count(r'\d')
url_df['num_special'] = url_str.str.count(r'[@?=&%]')
url_df['has_https'] = url_str.str.startswith('https').astype(int)
url_df['has_ip'] = url_str.str.contains(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', regex=True).astype(int)
url_df['has_at_symbol'] = url_str.str.contains('@').astype(int)

# Extract domain
def extract_domain(url):
    url = str(url)
    url = url.replace('https://', '').replace('http://', '')
    return url.split('/')[0]

url_df['domain'] = url_df['url'].apply(extract_domain)
url_df['domain_length'] = url_df['domain'].str.len()
url_df['subdomain_count'] = url_df['domain'].str.count(r'\.') - 1
url_df['subdomain_count'] = url_df['subdomain_count'].clip(lower=0)

print('Feature summary by class:')
feature_cols = ['url_length', 'num_dots', 'num_slashes', 'num_hyphens',
                'num_digits', 'num_special', 'has_https', 'has_ip',
                'has_at_symbol', 'domain_length', 'subdomain_count']
url_df.groupby('label_name')[feature_cols].mean().round(2).T

In [ ]:
# Feature comparison histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
plot_features = ['url_length', 'num_dots', 'num_slashes', 'num_digits', 'domain_length', 'subdomain_count']

for ax, feat in zip(axes.flat, plot_features):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = url_df[url_df['label'] == label]
        name = 'Legitimate' if label == 0 else 'Phishing'
        ax.hist(subset[feat], bins=30, alpha=0.6, label=name, color=color, edgecolor='black')
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('URL Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of URL features
corr_cols = feature_cols + ['label']
corr = url_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with label (phishing=1):')
print(corr['label'].drop('label').sort_values(ascending=False).round(3))

In [ ]:
# Top-level domain analysis
url_df['tld'] = url_df['domain'].apply(lambda d: d.split('.')[-1] if '.' in str(d) else '')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, title in [(axes[0], 0, 'Legitimate - Top TLDs'), (axes[1], 1, 'Phishing - Top TLDs')]:
    subset = url_df[url_df['label'] == label]
    top_tlds = subset['tld'].value_counts().head(15)
    ax.barh(top_tlds.index[::-1], top_tlds.values[::-1], color='#3498db' if label == 0 else '#e74c3c')
    ax.set_title(title)
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Character frequency analysis
def char_frequency(urls):
    all_chars = ''.join(urls.astype(str))
    return Counter(all_chars)

legit_chars = char_frequency(url_df[url_df['label'] == 0]['url'])
phish_chars = char_frequency(url_df[url_df['label'] == 1]['url'])

legit_total = sum(legit_chars.values())
phish_total = sum(phish_chars.values())

all_chars_set = sorted(set(legit_chars.keys()) | set(phish_chars.keys()))
special_chars = [c for c in all_chars_set if not c.isalnum()]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(special_chars))
width = 0.35

legit_freq = [legit_chars.get(c, 0) / legit_total * 100 for c in special_chars]
phish_freq = [phish_chars.get(c, 0) / phish_total * 100 for c in special_chars]

ax.bar(x - width/2, legit_freq, width, label='Legitimate', color='#2ecc71')
ax.bar(x + width/2, phish_freq, width, label='Phishing', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels([repr(c) for c in special_chars], rotation=45, ha='right')
ax.set_ylabel('Frequency (%)')
ax.set_title('Special Character Frequency in URLs')
ax.legend()
plt.tight_layout()
plt.show()

## 2. HTML Content Dataset

In [ ]:
# Load HTML dataset — detect folder names dynamically
html_dir = RAW_DIR / 'html_content'

html_records = []
if html_dir.exists():
    for subdir in html_dir.iterdir():
        if not subdir.is_dir():
            continue
        folder_name = subdir.name
        if 'phishing' in folder_name.lower():
            label_val = 1
        elif 'genuine' in folder_name.lower():
            label_val = 0
        else:
            print(f'  Skipping unknown folder: {folder_name}')
            continue
        files = list(subdir.iterdir())
        print(f'  {folder_name}: {len(files)} files')
        for fpath in files:
            if fpath.is_file():
                try:
                    content = fpath.read_text(errors='ignore')
                    html_records.append({
                        'filename': fpath.stem,
                        'file_size': fpath.stat().st_size,
                        'html_content': content,
                        'label': label_val,
                    })
                except Exception:
                    pass
else:
    print(f'WARNING: {html_dir} not found. Run scripts/download_data.py first.')

html_df = pd.DataFrame(html_records)
HAS_HTML = len(html_df) > 0

if HAS_HTML:
    html_df['label_name'] = html_df['label'].map({0: 'Legitimate', 1: 'Phishing'})
    print(f'\nTotal HTML files: {len(html_df)}')
    display(html_df.head())
else:
    print('No HTML files loaded. Skipping HTML analysis sections.')

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # HTML class distribution
    counts = html_df['label_name'].value_counts()
    colors = ['#2ecc71' if c == 'Legitimate' else '#e74c3c' for c in counts.index]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values, color=colors, edgecolor='black')
    ax.set_title('HTML Dataset - Class Distribution')
    ax.set_ylabel('Count')
    for i, (label, count) in enumerate(counts.items()):
        ax.text(i, count + 10, str(count), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # HTML content analysis
    html_df['content_length'] = html_df['html_content'].str.len()
    html_df['num_tags'] = html_df['html_content'].str.count(r'<[a-zA-Z]')
    html_df['num_scripts'] = html_df['html_content'].str.count(r'<script', case=False)
    html_df['num_forms'] = html_df['html_content'].str.count(r'<form', case=False)
    html_df['num_inputs'] = html_df['html_content'].str.count(r'<input', case=False)
    html_df['num_links'] = html_df['html_content'].str.count(r'<a\s', case=False)
    html_df['num_iframes'] = html_df['html_content'].str.count(r'<iframe', case=False)
    html_df['num_images'] = html_df['html_content'].str.count(r'<img', case=False)
    html_df['has_password_field'] = html_df['html_content'].str.contains(r'type=["\']password', case=False, regex=True).astype(int)
    html_df['has_login_text'] = html_df['html_content'].str.contains(r'login|sign.?in|log.?in', case=False, regex=True).astype(int)
    html_df['num_external_links'] = html_df['html_content'].str.count(r'https?://', case=False)

    print('HTML feature summary by class:')
    html_features = ['content_length', 'num_tags', 'num_scripts', 'num_forms',
                     'num_inputs', 'num_links', 'num_iframes', 'num_images',
                     'has_password_field', 'has_login_text', 'num_external_links']
    display(html_df.groupby('label_name')[html_features].mean().round(1).T)

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # HTML feature distributions
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    plot_feats = ['content_length', 'num_tags', 'num_scripts', 'num_forms', 'num_inputs', 'num_links']

    for ax, feat in zip(axes.flat, plot_feats):
        for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
            subset = html_df[html_df['label'] == label]
            name = 'Legitimate' if label == 0 else 'Phishing'
            data = subset[feat].clip(upper=subset[feat].quantile(0.95))
            ax.hist(data, bins=30, alpha=0.6, label=name, color=color, edgecolor='black')
        ax.set_title(feat)
        ax.legend(fontsize=8)

    plt.suptitle('HTML Feature Distributions by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # HTML content correlation with label
    corr_cols_html = html_features + ['label']
    corr_html = html_df[corr_cols_html].corr()

    print('Correlation with label (phishing=1):')
    print(corr_html['label'].drop('label').sort_values(ascending=False).round(3))
    print()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_html, dtype=bool))
    sns.heatmap(corr_html, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, square=True)
    ax.set_title('HTML Features - Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # Extract visible text from HTML
    from bs4 import BeautifulSoup

    def extract_text(html):
        try:
            soup = BeautifulSoup(str(html), 'lxml')
            for tag in soup(['script', 'style', 'meta', 'noscript', 'head']):
                tag.decompose()
            text = soup.get_text(separator=' ')
            return re.sub(r'\s+', ' ', text).strip()
        except Exception:
            return ''

    print('Extracting visible text (this may take a minute)...')
    html_df['visible_text'] = html_df['html_content'].apply(extract_text)
    html_df['text_length'] = html_df['visible_text'].str.len()
    html_df['word_count'] = html_df['visible_text'].str.split().str.len().fillna(0).astype(int)

    print('Visible text statistics:')
    display(html_df.groupby('label_name')[['text_length', 'word_count']].describe().round(0))

In [ ]:
if not HAS_HTML:
    print('Skipped — no HTML data available.')
else:
    # File size distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = html_df[html_df['label'] == label]
        name = 'Legitimate' if label == 0 else 'Phishing'
        data = subset['file_size'] / 1024  # KB
        data_clipped = data.clip(upper=data.quantile(0.95))
        axes[0].hist(data_clipped, bins=30, alpha=0.6, label=name, color=color, edgecolor='black')

    axes[0].set_title('HTML File Size Distribution')
    axes[0].set_xlabel('File Size (KB)')
    axes[0].legend()

    sns.boxplot(data=html_df, x='label_name', y=html_df['file_size']/1024, ax=axes[1],
                palette={'Legitimate': '#2ecc71', 'Phishing': '#e74c3c'},
                showfliers=False)
    axes[1].set_title('HTML File Size by Class (outliers hidden)')
    axes[1].set_ylabel('File Size (KB)')
    axes[1].set_xlabel('')

    plt.tight_layout()
    plt.show()

## 3. Screenshot Dataset

In [ ]:
# Load screenshot metadata — detect folder names dynamically
screenshot_dir = RAW_DIR / 'screenshots'
image_exts = {'.png', '.jpg', '.jpeg', '.webp'}

img_records = []
if screenshot_dir.exists():
    for subdir in screenshot_dir.iterdir():
        if not subdir.is_dir():
            continue
        folder_name = subdir.name
        if 'phishing' in folder_name.lower():
            label_val = 1
        elif 'genuine' in folder_name.lower():
            label_val = 0
        else:
            print(f'  Skipping unknown folder: {folder_name}')
            continue
        files = [f for f in subdir.iterdir() if f.suffix.lower() in image_exts]
        print(f'  {folder_name}: {len(files)} images')
        for fpath in files:
            img_records.append({
                'filename': fpath.stem,
                'image_path': str(fpath),
                'file_size': fpath.stat().st_size,
                'extension': fpath.suffix.lower(),
                'label': label_val,
            })
else:
    print(f'WARNING: {screenshot_dir} not found. Run scripts/download_data.py first.')

img_df = pd.DataFrame(img_records)
HAS_IMAGES = len(img_df) > 0

if HAS_IMAGES:
    img_df['label_name'] = img_df['label'].map({0: 'Legitimate', 1: 'Phishing'})
    print(f'\nTotal images: {len(img_df)}')
    display(img_df.head())
else:
    print('No screenshot files loaded. Skipping image analysis sections.')

In [ ]:
if not HAS_IMAGES:
    print('Skipped — no screenshot data available.')
else:
    # Screenshot class distribution
    counts = img_df['label_name'].value_counts()
    colors = ['#2ecc71' if c == 'Legitimate' else '#e74c3c' for c in counts.index]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values, color=colors, edgecolor='black')
    ax.set_title('Screenshot Dataset - Class Distribution')
    ax.set_ylabel('Count')
    for i, (label, count) in enumerate(counts.items()):
        ax.text(i, count + 10, str(count), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_IMAGES:
    print('Skipped — no screenshot data available.')
else:
    # Image dimensions and file sizes
    print('Reading image dimensions (sampling up to 500 per class)...')
    sample_records = []
    for label in [0, 1]:
        label_subset = img_df[img_df['label'] == label]
        n_sample = min(500, len(label_subset))
        if n_sample == 0:
            continue
        subset = label_subset.sample(n=n_sample, random_state=42)
        for _, row in subset.iterrows():
            try:
                img = Image.open(row['image_path'])
                w, h = img.size
                sample_records.append({
                    'filename': row['filename'],
                    'width': w,
                    'height': h,
                    'aspect_ratio': w / h if h > 0 else 0,
                    'file_size_kb': row['file_size'] / 1024,
                    'label': label,
                    'label_name': row['label_name'],
                })
            except Exception:
                pass

    img_meta = pd.DataFrame(sample_records)
    print(f'Sampled {len(img_meta)} images')

    print('\nImage dimension statistics:')
    display(img_meta.groupby('label_name')[['width', 'height', 'aspect_ratio', 'file_size_kb']].describe().round(1))

In [ ]:
if not HAS_IMAGES:
    print('Skipped — no screenshot data available.')
else:
    # Image size distributions
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = img_meta[img_meta['label'] == label]
        if len(subset) == 0:
            continue
        name = 'Legitimate' if label == 0 else 'Phishing'
        axes[0].hist(subset['width'], bins=30, alpha=0.6, label=name, color=color, edgecolor='black')
        axes[1].hist(subset['height'], bins=30, alpha=0.6, label=name, color=color, edgecolor='black')
        axes[2].hist(subset['file_size_kb'].clip(upper=subset['file_size_kb'].quantile(0.95)),
                     bins=30, alpha=0.6, label=name, color=color, edgecolor='black')

    axes[0].set_title('Image Width'); axes[0].set_xlabel('Pixels'); axes[0].legend()
    axes[1].set_title('Image Height'); axes[1].set_xlabel('Pixels'); axes[1].legend()
    axes[2].set_title('File Size'); axes[2].set_xlabel('KB'); axes[2].legend()

    plt.suptitle('Screenshot Dimensions by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_IMAGES:
    print('Skipped — no screenshot data available.')
else:
    # Sample screenshots grid
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))

    for row, (label, title) in enumerate([(0, 'Legitimate'), (1, 'Phishing')]):
        label_subset = img_df[img_df['label'] == label]
        n_sample = min(5, len(label_subset))
        if n_sample == 0:
            for col in range(5):
                axes[row, col].text(0.5, 0.5, 'No data', ha='center', va='center')
                axes[row, col].axis('off')
            continue
        subset = label_subset.sample(n=n_sample, random_state=42)
        for col, (_, sample) in enumerate(subset.iterrows()):
            try:
                img = Image.open(sample['image_path']).convert('RGB')
                axes[row, col].imshow(img)
                axes[row, col].set_title(f"{title}\n{sample['filename'][:20]}", fontsize=9)
            except Exception:
                axes[row, col].text(0.5, 0.5, 'Error', ha='center', va='center')
            axes[row, col].axis('off')
        # Blank remaining axes if fewer than 5 samples
        for col in range(n_sample, 5):
            axes[row, col].axis('off')

    plt.suptitle('Sample Screenshots', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_IMAGES:
    print('Skipped — no screenshot data available.')
else:
    # Average color analysis
    print('Computing average colors...')
    color_records = []
    for _, row in img_meta.iterrows():
        try:
            matches = img_df[img_df['filename'] == row['filename']]
            if len(matches) == 0:
                continue
            img = Image.open(matches['image_path'].iloc[0]).convert('RGB')
            img_arr = np.array(img.resize((64, 64)))
            color_records.append({
                'mean_r': img_arr[:,:,0].mean(),
                'mean_g': img_arr[:,:,1].mean(),
                'mean_b': img_arr[:,:,2].mean(),
                'brightness': img_arr.mean(),
                'std': img_arr.std(),
                'label': row['label'],
                'label_name': row['label_name'],
            })
        except Exception:
            pass

    color_df = pd.DataFrame(color_records)
    if len(color_df) > 0:
        print('\nAverage color statistics:')
        display(color_df.groupby('label_name')[['mean_r', 'mean_g', 'mean_b', 'brightness', 'std']].mean().round(1))
    else:
        print('Could not compute color statistics.')

In [ ]:
if not HAS_IMAGES or 'color_df' not in dir() or len(color_df) == 0:
    print('Skipped — no color data available.')
else:
    # Brightness and contrast distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = color_df[color_df['label'] == label]
        if len(subset) == 0:
            continue
        name = 'Legitimate' if label == 0 else 'Phishing'
        axes[0].hist(subset['brightness'], bins=30, alpha=0.6, label=name, color=color, edgecolor='black')
        axes[1].hist(subset['std'], bins=30, alpha=0.6, label=name, color=color, edgecolor='black')

    axes[0].set_title('Mean Brightness'); axes[0].legend()
    axes[1].set_title('Pixel Std Dev (Contrast)'); axes[1].legend()

    plt.suptitle('Screenshot Color Properties by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Cross-Modal Alignment Check

In [ ]:
# Check dataset sizes and alignment
print('Dataset sizes:')
print(f'  URLs:        {len(url_df):,} rows')
print(f'  HTML files:  {len(html_df):,} files')
print(f'  Screenshots: {len(img_df):,} files')
print()

# Check label consistency across datasets
print('Label distribution comparison:')
summary_data = {'URL': url_df['label'].value_counts().sort_index()}
if HAS_HTML:
    summary_data['HTML'] = html_df['label'].value_counts().sort_index()
if HAS_IMAGES:
    summary_data['Screenshot'] = img_df['label'].value_counts().sort_index()
summary = pd.DataFrame(summary_data).rename(index={0: 'Legitimate', 1: 'Phishing'})
print(summary)
print()

# Check filename overlap
url_indices = set(str(i) for i in range(len(url_df)))
html_filenames = set(html_df['filename']) if HAS_HTML else set()
img_filenames = set(img_df['filename']) if HAS_IMAGES else set()

print('Filename-based alignment:')
if HAS_HTML:
    print(f'  HTML filenames that are numeric indices: {len(html_filenames & url_indices)}/{len(html_filenames)}')
if HAS_IMAGES:
    print(f'  Screenshot filenames that are numeric indices: {len(img_filenames & url_indices)}/{len(img_filenames)}')
if HAS_HTML and HAS_IMAGES:
    print(f'  HTML-Screenshot filename overlap: {len(html_filenames & img_filenames)}/{max(len(html_filenames), len(img_filenames))}')

In [ ]:
# Visualize dataset overlap
if not (HAS_HTML and HAS_IMAGES):
    print('Need all three datasets for Venn diagram.')
    print(f'  URL indices: {len(url_indices)}')
    if HAS_HTML: print(f'  HTML files: {len(html_filenames)}')
    if HAS_IMAGES: print(f'  Screenshots: {len(img_filenames)}')
else:
    try:
        from matplotlib_venn import venn3
        fig, ax = plt.subplots(figsize=(8, 6))
        venn3(
            [url_indices, html_filenames, img_filenames],
            set_labels=('URL indices', 'HTML files', 'Screenshots'),
            ax=ax
        )
        ax.set_title('Dataset Alignment (by filename/index)')
        plt.show()
    except ImportError:
        print('Install matplotlib-venn for Venn diagram: pip install matplotlib-venn')
        print(f'  URL indices: {len(url_indices)}')
        print(f'  HTML files: {len(html_filenames)}')
        print(f'  Screenshots: {len(img_filenames)}')
        print(f'  All three overlap: {len(url_indices & html_filenames & img_filenames)}')

## 5. Summary & Key Findings

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print()
print(f'Total URL samples:  {len(url_df)}')
print(f'  Legitimate:       {(url_df["label"] == 0).sum()}')
print(f'  Phishing:         {(url_df["label"] == 1).sum()}')
print(f'  Balance:          {url_df["label"].mean():.1%} phishing')
print()
print('Modality availability:')
print(f'  URLs:         {len(url_df)} (100%)')
print(f'  HTML:         {len(html_df)} ({len(html_df)/max(len(url_df),1)*100:.0f}%)')
print(f'  Screenshots:  {len(img_df)} ({len(img_df)/max(len(url_df),1)*100:.0f}%)')
print()
print('Top discriminative URL features (by |correlation| with label):')
url_corr = url_df[feature_cols + ['label']].corr()['label'].drop('label').abs().sort_values(ascending=False)
for feat, val in url_corr.head(5).items():
    print(f'  {feat:20s}  |corr| = {val:.3f}')
print()
if HAS_HTML and 'html_features' in dir():
    print('Top discriminative HTML features (by |correlation| with label):')
    html_corr = html_df[html_features + ['label']].corr()['label'].drop('label').abs().sort_values(ascending=False)
    for feat, val in html_corr.head(5).items():
        print(f'  {feat:20s}  |corr| = {val:.3f}')
    print()
print('=' * 60)